# Modeling ประมาณองค์ประกอบร่างกายจากค่าวัด

ใช้ชุดข้อมูลจาก `src/real_label.py` (NHANES 2017-2018 ผู้ใหญ่ 18-59 ปีที่มีผล DXA ครบ) ประมาณ Fat % กับมวลกล้ามเนื้อ (Lean mass, ALM, Trunk lean) จากค่าที่วัดง่าย โดยไม่ต้องสแกน DXA แล้วต่อยอดไปคัดกรองคนที่มีมวลกล้ามเนื้อต่ำ

In [1]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, roc_auc_score, average_precision_score, roc_curve
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold, cross_val_predict, cross_val_score, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier
warnings.filterwarnings("ignore")

os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
sys.path.insert(0, "src")
import real_label as rl
df, n_adults = rl.build()
df.to_csv(rl.OUT, index=False)
BASIC, CIRC, LIFE = rl.BASIC, rl.CIRC, rl.LIFE
FEAT = BASIC + CIRC
TARGETS = ["FAT_PCT", "LEAN_KG", "ALM_KG", "TRUNK_KG"]
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df.LOW_HT2)
print(f"adults 18-59: {n_adults} -> with DXA + body measures: {len(df)} | train {len(train)} / test {len(test)}")

def score(y, p):
    return {"R2": r2_score(y, p), "MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p)))}

def ridge():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), RidgeCV(alphas=[0.1, 1, 10, 100]))

summary = {"n_adults": n_adults, "n": len(df), "n_train": len(train), "n_test": len(test),
           "prevalence": {"LOW_HT2": float(df.LOW_HT2.mean()), "LOW_FNIH": float(df.LOW_FNIH.mean()), "both": int((df.LOW_FNIH & df.LOW_HT2).sum())}}

adults 18-59: 3515 -> with DXA + body measures: 2310 | train 1848 / test 462


## 1. DXA หายในคนที่น้ำหนักมาก (selection bias)

In [2]:
bins = pd.cut(df.BMXWT, [0, 60, 80, 100, 120, 300], labels=["<60", "60-80", "80-100", "100-120", ">=120"])
print(df.groupby(bins, observed=True).agg(n=("SEQN", "size"), fat_pct=("FAT_PCT", "mean"), lean=("LEAN_KG", "mean")).round(1))
print("low muscle (ALM/ht^2):", int(df.LOW_HT2.sum()), "| (ALM/BMI):", int(df.LOW_FNIH.sum()), "| both:", int((df.LOW_FNIH & df.LOW_HT2).sum()))

           n  fat_pct  lean
BMXWT                      
<60      413     30.0  35.7
60-80    919     31.8  46.1
80-100   639     33.9  56.6
100-120  240     36.9  66.2
>=120     99     40.5  75.2
low muscle (ALM/ht^2): 233 | (ALM/BMI): 234 | both: 23


## 2. ก่อนเทรน ทายค่าเฉลี่ยกับใช้สูตร Deurenberg (1991)

สูตรคือ `Fat% = 1.20×BMI + 0.23×อายุ − 10.8×(ชาย=1) − 5.4` ใช้เป็นเส้นฐานไว้เทียบกับ model

In [3]:
before = {t: score(test[t], np.full(len(test), train[t].mean())) for t in TARGETS}
deur = 1.20 * test.BMXBMI + 0.23 * test.RIDAGEYR - 10.8 * (test.RIAGENDR == 1) - 5.4
deur_score = score(test.FAT_PCT, deur)
display(pd.DataFrame(before).T.round(3)); print("Deurenberg (Fat %):", {k: round(v, 3) for k, v in deur_score.items()})
summary["before"], summary["deurenberg"] = before, deur_score

,R2,MAE,RMSE
FAT_PCT,-0.004,7.199,8.687
LEAN_KG,-0.000,10.485,12.789
ALM_KG,-0.000,5.260,6.465
TRUNK_KG,-0.000,5.104,6.258


Deurenberg (Fat %): {'R2': 0.529, 'MAE': 4.612, 'RMSE': 5.949}


## 3. ลอง 3 โมเดล Ridge, Random Forest, XGBoost

ปรับค่าตั้งของแต่ละโมเดลด้วย 5-fold CV บนชุดสอนอย่างเดียว แล้ววัดบน Test พร้อม CV ± std และเวลาที่ใช้

In [4]:
xgb_grid = {"n_estimators": [100, 200, 300], "max_depth": [2, 3, 4], "learning_rate": [0.03, 0.05, 0.1]}
rf_grid = {"n_estimators": [200, 400], "max_depth": [6, 10, None], "min_samples_leaf": [1, 3, 5]}
kf = KFold(5, shuffle=True, random_state=42)
tuned_xgb, mc_rows, models3, best = {}, [], {}, {}
for t in TARGETS:
    gx = GridSearchCV(XGBRegressor(random_state=42), xgb_grid, cv=5, scoring="r2", n_jobs=-1).fit(train[FEAT], train[t])
    gr = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), rf_grid, cv=5, scoring="r2", n_jobs=1).fit(train[FEAT], train[t])
    tuned_xgb[t] = gx.best_estimator_
    best[t] = {"XGBoost": gx.best_params_, "Random Forest": gr.best_params_}
    models3[t] = {"Ridge": ridge(), "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1, **gr.best_params_),
                  "XGBoost": XGBRegressor(random_state=42, **gx.best_params_)}
    for name, m in models3[t].items():
        cv = cross_val_score(m, train[FEAT], train[t], cv=kf, scoring="r2")
        t0 = time.time(); m.fit(train[FEAT], train[t]); fit_s = time.time() - t0
        t0 = time.time(); p = m.predict(test[FEAT]); pred_ms = (time.time() - t0) * 1e6 / len(test)
        if name == "Ridge": best[t]["Ridge alpha"] = float(m[-1].alpha_)
        mc_rows.append({"Target": t, "Model": name, **{"Test " + k: v for k, v in score(test[t], p).items()},
                        "CV R2": cv.mean(), "CV std": cv.std(), "Fit s": fit_s, "Predict ms/1000": pred_ms})
df_mc = pd.DataFrame(mc_rows); summary["best_params"] = best
print("ค่าที่ชนะจาก GridSearchCV (Ridge คือ alpha จาก RidgeCV)")
for t in TARGETS: print(t, best[t])
df_mc.round(3)

ค่าที่ชนะจาก GridSearchCV (Ridge คือ alpha จาก RidgeCV)
FAT_PCT {'XGBoost': {'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 200}, 'Random Forest': {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}, 'Ridge alpha': 0.1}
LEAN_KG {'XGBoost': {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}, 'Random Forest': {'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 400}, 'Ridge alpha': 0.1}
ALM_KG {'XGBoost': {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}, 'Random Forest': {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 400}, 'Ridge alpha': 0.1}
TRUNK_KG {'XGBoost': {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}, 'Random Forest': {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}, 'Ridge alpha': 0.1}


,Target,Model,Test R2,Test MAE,Test RMSE,CV R2,CV std,Fit s,Predict ms/1000
0,FAT_PCT,Ridge,0.824,2.879,3.634,0.825,0.017,0.006,4.329
1,FAT_PCT,Random Forest,0.835,2.753,3.527,0.845,0.007,0.612,185.118
2,FAT_PCT,XGBoost,0.837,2.750,3.498,0.846,0.010,0.291,11.929
3,LEAN_KG,Ridge,0.949,2.240,2.898,0.953,0.004,0.006,0.000
4,LEAN_KG,Random Forest,0.948,2.246,2.908,0.948,0.006,0.669,198.448
5,LEAN_KG,XGBoost,0.951,2.187,2.845,0.950,0.005,0.412,13.053
6,ALM_KG,Ridge,0.928,1.340,1.739,0.931,0.010,0.006,2.168
7,ALM_KG,Random Forest,0.925,1.343,1.773,0.918,0.012,0.970,214.161
8,ALM_KG,XGBoost,0.930,1.301,1.714,0.924,0.011,0.424,8.672
9,TRUNK_KG,Ridge,0.928,1.288,1.678,0.933,0.004,0.007,2.160


## 4. ทดสอบกับคนน้ำหนักมาก

สอนด้วยคนต่ำกว่า 90 kg แล้วทายคน 100 kg ขึ้นไป จำลองสถานการณ์จริงที่ DXA มักหายในคนน้ำหนักมาก

In [5]:
lo, hi = df[df.BMXWT < 90], df[df.BMXWT >= 100]
ex_rows = []
for t in TARGETS:
    for name, m in models3[t].items():
        p = clone(m).fit(lo[FEAT], lo[t]).predict(hi[FEAT])
        ex_rows.append({"Target": t, "Model": name, "n heavy": len(hi), "Heavy MAE": mean_absolute_error(hi[t], p), "Heavy bias": float(np.mean(p - hi[t]))})
df_ex = pd.DataFrame(ex_rows); df_ex.round(2)

,Target,Model,n heavy,Heavy MAE,Heavy bias
0,FAT_PCT,Ridge,339,4.70,4.32
1,FAT_PCT,Random Forest,339,3.71,-2.59
2,FAT_PCT,XGBoost,339,3.47,-1.97
3,LEAN_KG,Ridge,339,3.16,1.12
4,LEAN_KG,Random Forest,339,12.98,-12.97
5,LEAN_KG,XGBoost,339,14.08,-14.06
6,ALM_KG,Ridge,339,1.95,0.91
7,ALM_KG,Random Forest,339,5.19,-5.14
8,ALM_KG,XGBoost,339,6.22,-6.17
9,TRUNK_KG,Ridge,339,1.77,-0.02


## 5. เลือกโมเดล

| โจทย์ | เลือก | เหตุผล |
|---|---|---|
| Fat % | **XGBoost** | แม่นสุดทั้ง CV และ Test และพลาดน้อยสุดในคนหนัก |
| Lean mass, ALM, Trunk lean | **Ridge** | แม่นเท่าโมเดลอื่น (ต่างไม่เกิน 0.005 ใน CV) แต่ง่ายสุด เร็วสุด ตีความได้ และทนคนหนักได้ดีกว่ามากกว่า 4 เท่า |
| ไม่เลือก | Random Forest | ไม่ชนะข้อไหนเลย ช้ากว่า ไม่แม่นกว่า และ extrapolate ไม่ได้ |

In [6]:
final_name = {"FAT_PCT": "XGBoost", "LEAN_KG": "Ridge", "ALM_KG": "Ridge", "TRUNK_KG": "Ridge"}
def make_final(t):
    return clone(models3[t][final_name[t]])
final = {t: make_final(t).fit(train[FEAT], train[t]) for t in TARGETS}
summary["model_comparison"] = {"regression": df_mc.to_dict("records"), "extrapolation": df_ex.to_dict("records"), "final": final_name}
tbl = df_mc.merge(df_ex, on=["Target", "Model"])
tbl["Chosen"] = [final_name[r.Target] == r.Model for r in tbl.itertuples()]
tbl[["Target", "Model", "Test R2", "Test MAE", "CV R2", "Heavy MAE", "Predict ms/1000", "Chosen"]].round(3)

,Target,Model,Test R2,Test MAE,CV R2,Heavy MAE,Predict ms/1000,Chosen
0,FAT_PCT,Ridge,0.824,2.879,0.825,4.704,4.329,False
1,FAT_PCT,Random Forest,0.835,2.753,0.845,3.708,185.118,False
2,FAT_PCT,XGBoost,0.837,2.750,0.846,3.472,11.929,True
3,LEAN_KG,Ridge,0.949,2.240,0.953,3.156,0.000,True
4,LEAN_KG,Random Forest,0.948,2.246,0.948,12.984,198.448,False
5,LEAN_KG,XGBoost,0.951,2.187,0.950,14.076,13.053,False
6,ALM_KG,Ridge,0.928,1.340,0.931,1.945,2.168,True
7,ALM_KG,Random Forest,0.925,1.343,0.918,5.194,214.161,False
8,ALM_KG,XGBoost,0.930,1.301,0.924,6.224,8.672,False
9,TRUNK_KG,Ridge,0.928,1.288,0.933,1.773,2.160,True


## 6. ผลสุดท้าย ก่อนเทรนเทียบหลังเทรน (โมเดลที่เลือก) พร้อมช่วงความเชื่อมั่น (bootstrap 1,000 รอบ)

In [7]:
after = {t: score(test[t], final[t].predict(test[FEAT])) for t in TARGETS}
rng = np.random.default_rng(42); boot = {}
preds = {t: final[t].predict(test[FEAT]) for t in TARGETS}
for t in TARGETS:
    r2s, maes, gain = [], [], []
    for _ in range(1000):
        i = rng.integers(0, len(test), len(test)); y, p = test[t].values[i], preds[t][i]
        r2s.append(r2_score(y, p)); maes.append(mean_absolute_error(y, p))
        gain.append(1 - mean_absolute_error(y, p) / mean_absolute_error(y, np.full(len(y), train[t].mean())))
    boot[t] = {"R2": np.percentile(r2s, [2.5, 97.5]).tolist(), "MAE": np.percentile(maes, [2.5, 97.5]).tolist(), "MAE_reduction": np.percentile(gain, [2.5, 97.5]).tolist()}
res = pd.DataFrame({t: {"model": final_name[t], "R2 before": before[t]["R2"], "R2 after": after[t]["R2"], "MAE before": before[t]["MAE"], "MAE after": after[t]["MAE"],
                        "MAE cut %": round((1 - after[t]["MAE"] / before[t]["MAE"]) * 100)} for t in TARGETS}).T
summary["after"], summary["bootstrap_ci"] = after, {"regression": boot}
display(res.round(3)); print(json.dumps(boot))

,model,R2 before,R2 after,MAE before,MAE after,MAE cut %
FAT_PCT,XGBoost,-0.003653,0.837272,7.198594,2.750306,62
LEAN_KG,Ridge,-0.000003,0.948655,10.484637,2.240133,79
ALM_KG,Ridge,-0.000018,0.927639,5.260064,1.339721,75
TRUNK_KG,Ridge,-0.000006,0.928083,5.103641,1.287848,75


{"FAT_PCT": {"R2": [0.8113570409332539, 0.8587216648117422], "MAE": [2.564568140973457, 2.9485307604814013], "MAE_reduction": [0.5837569333710388, 0.6494200190443725]}, "LEAN_KG": {"R2": [0.9374818586465526, 0.9577077986466067], "MAE": [2.076787086134078, 2.410582106876481], "MAE_reduction": [0.7656125611598739, 0.8059420362720058]}, "ALM_KG": {"R2": [0.9142822172644308, 0.9395337500666656], "MAE": [1.2394590989606185, 1.4430887482738084], "MAE_reduction": [0.7226530518164352, 0.7682672535491272]}, "TRUNK_KG": {"R2": [0.914266552397578, 0.9401644544048201], "MAE": [1.1931972459613636, 1.3871392677694314], "MAE_reduction": [0.7225531789055524, 0.7697629019747301]}}
